# Step 5: remake the lost corruption files and finish step 4

Drive filled up (15 GB), so most corruption files from notebook 3 never got saved: HAM LoRA after the first 3 corruptions, and all of Fitz. This remakes them on Colab's own disk (not Drive), runs `calibrate_extra.py` on each setting, and saves only the small result files to `MyDrive/grsats/results_extra`. HAM LP is already done from notebook 4.

1. Runtime > Change runtime type > T4 GPU.
2. Run the cells in order and keep the tab open. It takes several hours (HAM LoRA first, then Fitz).
3. If Colab disconnects, run everything again. Settings whose results are already on Drive get skipped, but a setting that was half done starts over.
4. When it's done, download everything in `MyDrive/grsats/results_extra` into `code/results/` on the laptop.

Each setting prints `largest wg_nll difference from results/...`. It should be tiny (about 1e-6 or less), which means the remade files match the ones `calibrate.py` used.

In [ ]:
import os
import torch
from google.colab import drive, userdata

assert torch.cuda.is_available(), "no gpu, change the runtime type to T4"
print(torch.cuda.get_device_name(0))

drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/adipatel11/gr-sats-term-paper.git /content/repo || git -C /content/repo pull -q
%cd /content/repo/code
!pip install -q -r requirements.txt

# same medmnist-c install as notebook 3
!apt-get -qq update
!apt-get -qq install -y libmagickwand-dev > /dev/null
!pip install -q --no-deps medmnistc

In [ ]:
# unzip to the colab disk, reading from drive is really slow
ham = "../data/HAM10000/dataverse_files"
!mkdir -p {ham}
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_1.zip -d {ham}/HAM10000_images_part_1
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_2.zip -d {ham}/HAM10000_images_part_2
!unzip -qn /content/drive/MyDrive/grsats/finalfitz17k.zip -d ../data

# missing image files should be 0 for both
!python check_data.py
!python -m pytest -q

In [ ]:
# drive is full, so this time everything big stays on colab's disk. copy the inputs over
# (features, probes, lora weights, and the 3 ham lora corruption files that did reach drive)
drive_out = "/content/drive/MyDrive/grsats/outputs"
local = "/content/outputs"
!mkdir -p {local}
!rsync -a --exclude "ham_c_*_frozen.npz" --exclude "*.tmp.npz" {drive_out}/ {local}/
os.environ["GRSATS_OUT"] = local
!ls {local} | wc -l
!df -h /content | tail -1

In [ ]:
# one dataset at a time: remake the missing corruption files, then run calibrate_extra.py on
# each setting (fitz runs its two at once) and copy the results to drive right away.
# fitz uses --models both so each image only gets corrupted once for all 8 models
res = "/content/drive/MyDrive/grsats/results_extra"
!mkdir -p {res}
files = ["extra.csv", "detail.csv", "reliability.csv", "scores.csv", "perimage.npz"]

for ds, settings in [("ham", ["lora"]), ("fitz", ["frozen", "lora"])]:
    todo = [m for m in settings if not os.path.exists(f"{res}/{ds}_{m}_extra.csv")]
    if not todo:
        print(f"{ds} already done, skipping")
        continue
    which = todo[0] if len(todo) == 1 else "both"
    !python corrupt_extract.py --dataset {ds} --models {which}

    runs = " & ".join(f"python calibrate_extra.py --dataset {ds} --models {m} > {local}/{ds}_{m}.log 2>&1" for m in todo)
    !{runs}; wait
    for m in todo:
        print(f"\n===== {ds} {m} =====")
        !grep -v "fold . done" {local}/{ds}_{m}.log
        for f in files:
            !cp {local}/{ds}_{m}_{f} {res}/

!ls -la {res}

In [ ]:
# make sure everything copied to drive is really uploaded before the session ends
drive.flush_and_unmount()
print("done, download MyDrive/grsats/results_extra into code/results/")